# 01 - Generacion de los 54 samples del grid

Aplica los filtros parametrizados a `users.csv` y genera un sample por cada configuracion. NO genera features (eso es 02a/02b).

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[0]))

import pandas as pd
import numpy as np
from time import time
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from scripts import config as cfg
from scripts import grid_utils
from scripts import sample_generation as sg

np.random.seed(cfg.RANDOM_SEED)


In [2]:
# [EXEC] Cargar users_raw una sola vez
t0 = time()
users_raw, reference_date = sg.load_users_raw()
print(f'users_raw: {users_raw.shape}')
print(f'REFERENCE_DATE: {reference_date}')
print(f'Carga: {time()-t0:.1f}s')


users_raw: (1164568, 5)
REFERENCE_DATE: 2026-04-04 18:23:37
Carga: 4.5s


In [3]:
# [EXEC] Generar los 54 samples del grid
grid = grid_utils.generate_grid()
print(f'Generando {len(grid)} samples...\n')

results = []
for i, conf in enumerate(grid):
    t0 = time()
    sample = sg.generate_sample_for_config(
        users_raw, reference_date,
        cutoff=conf['cutoff'], spike=conf['spike'], min_logins=conf['min_logins'],
    )
    out_path = sg.save_sample(sample, conf['id'])
    elapsed = time() - t0

    n = len(sample)
    ch14 = float(sample['churn_14d'].mean()) if 'churn_14d' in sample.columns else None
    ch30 = float(sample['churn_30d'].mean()) if 'churn_30d' in sample.columns else None

    results.append({
        'id': conf['id'], 'block': conf['block'],
        'cutoff': conf['cutoff'], 'spike': conf['spike'], 'min_logins': conf['min_logins'],
        'is_baseline': conf['is_baseline'],
        'n_users': n,
        'churn_14d_rate': ch14, 'churn_30d_rate': ch30,
        'time_s': round(elapsed, 2),
    })
    ch30s = f'{ch30:.3f}' if ch30 is not None else 'n/a'
    ch14s = f'{ch14:.3f}' if ch14 is not None else 'n/a'
    print(f"[{i+1:2d}/{len(grid)}] {conf['id']:>4} cutoff={conf['cutoff']:>3} spike={conf['spike']:>2} ml={conf['min_logins']:>2} N={n:>7,} ch14={ch14s} ch30={ch30s} t={elapsed:.1f}s")

print(f'\n[OK] Total: {len(results)} samples generados')


Generando 54 samples...

[ 1/54]  L01 cutoff= 30 spike= 3 ml= 2 N= 15,832 ch14=0.612 ch30=1.000 t=0.2s
[ 2/54]  L02 cutoff= 30 spike= 3 ml= 5 N= 11,813 ch14=0.558 ch30=1.000 t=0.1s
[ 3/54]  L03 cutoff= 30 spike= 3 ml=10 N=  7,943 ch14=0.507 ch30=1.000 t=0.1s
[ 4/54]  L04 cutoff= 30 spike= 7 ml= 2 N= 21,358 ch14=0.712 ch30=1.000 t=0.2s
[ 5/54]  L05 cutoff= 30 spike= 7 ml= 5 N= 14,568 ch14=0.642 ch30=1.000 t=0.1s
[ 6/54]  L06 cutoff= 30 spike= 7 ml=10 N=  9,321 ch14=0.580 ch30=1.000 t=0.1s
[ 7/54]  L07 cutoff= 30 spike=14 ml= 2 N= 31,474 ch14=0.805 ch30=1.000 t=0.2s
[ 8/54]  L08 cutoff= 30 spike=14 ml= 5 N= 19,454 ch14=0.732 ch30=1.000 t=0.2s
[ 9/54]  L09 cutoff= 30 spike=14 ml=10 N= 11,769 ch14=0.667 ch30=1.000 t=0.1s
[10/54]  L10 cutoff= 60 spike= 3 ml= 2 N= 22,475 ch14=0.447 ch30=0.654 t=0.2s
[11/54]  L11 cutoff= 60 spike= 3 ml= 5 N= 17,037 ch14=0.383 ch30=0.605 t=0.1s
[12/54]  L12 cutoff= 60 spike= 3 ml=10 N= 11,485 ch14=0.333 ch30=0.561 t=0.1s
[13/54]  L13 cutoff= 60 spike= 7 ml= 2 

In [4]:
# [ANALYSIS] Tabla resumen
results_df = pd.DataFrame(results)
print(f'Resumen:')
print(results_df.describe()[['n_users','churn_14d_rate','churn_30d_rate','time_s']])
print(f'\nBaseline:')
print(results_df[results_df['is_baseline']][['id','cutoff','spike','min_logins','n_users','churn_14d_rate','churn_30d_rate']].to_string(index=False))

results_df.to_parquet(cfg.STUDY_SAMPLES / '_grid_summary.parquet', index=False)
results_df.to_csv(cfg.STUDY_INFORMES / '01_grid_samples_summary.csv', index=False)


Resumen:
            n_users  churn_14d_rate  churn_30d_rate     time_s
count     54.000000       54.000000       54.000000  54.000000
mean   21058.351852        0.527457        0.680410   0.142037
std     9187.302062        0.257850        0.251364   0.024370
min     5828.000000        0.170344        0.287205   0.110000
25%    14650.000000        0.330441        0.472279   0.120000
50%    19380.000000        0.459060        0.625068   0.140000
75%    27905.000000        0.660756        1.000000   0.160000
max    44250.000000        1.000000        1.000000   0.210000

Baseline:
 id  cutoff  spike  min_logins  n_users  churn_14d_rate  churn_30d_rate
L32     120      7           5    25380        0.337155        0.470883


In [5]:
# [ANALYSIS] Sanity check baseline
baseline_row = results_df[results_df['is_baseline']].iloc[0]
expected_n = 25200
actual_n = baseline_row['n_users']
delta = actual_n - expected_n
pct = abs(delta) / expected_n * 100

if pct < 2:
    print(f'[OK] Baseline: N={actual_n:,} (esperado ~{expected_n}, diff={delta:+d} = {pct:.1f}%)')
    print('  Diferencia atribuible al filtro forense F6 del 02a (no replicado).')
elif pct < 5:
    print(f'[WARN] Baseline cerca: N={actual_n:,} vs esperado {expected_n} ({pct:.1f}%)')
else:
    print(f'[FAIL] Baseline DIVERGE: N={actual_n:,} vs esperado {expected_n} ({pct:.1f}%)')


[OK] Baseline: N=25,380 (esperado ~25200, diff=+180 = 0.7%)
  Diferencia atribuible al filtro forense F6 del 02a (no replicado).


In [6]:
# [ANALYSIS] Visualizacion
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# N por cutoff (bloque L)
g1 = results_df[results_df['block']=='L'].groupby('cutoff')['n_users'].agg(['min','median','max'])
g1.plot(kind='bar', ax=axes[0], title='N por cutoff (bloque L)')
axes[0].set_ylabel('N usuarios')

# Churn rates por config
results_df.sort_values('id').plot(x='id', y=['churn_14d_rate','churn_30d_rate'],
                                    ax=axes[1], title='Churn rates por config', rot=90)
axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(cfg.STUDY_INFORMES / '01_grid_samples_overview.png', dpi=120, bbox_inches='tight')
plt.show()


/var/folders/0f/3zq36dr1451_ttvt93n947ph0000gw/T/ipykernel_39709/156539275.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# [REPORT] 01_grid_samples_report.md
from datetime import datetime

report = f"""# Generacion del grid de samples - Sub-sesion 1a

**Fecha**: {datetime.now()}

## Resumen ejecutivo

- {len(results_df)} samples generados
- Tiempo total: {results_df['time_s'].sum():.1f}s
- N min: {results_df['n_users'].min():,}
- N medio: {results_df['n_users'].mean():.0f}
- N max: {results_df['n_users'].max():,}

## Baseline (config TFG = L32)

| Metric | Value |
|---|---|
| ID | {baseline_row['id']} |
| cutoff | {baseline_row['cutoff']} |
| spike | {baseline_row['spike']} |
| min_logins | {baseline_row['min_logins']} |
| N | {baseline_row['n_users']:,} |
| churn_14d_rate | {baseline_row['churn_14d_rate']:.4f} |
| churn_30d_rate | {baseline_row['churn_30d_rate']:.4f} |

## Sanity check

- N baseline: {baseline_row['n_users']:,} (esperado ~25,200 del TFG)
- Diff: {delta:+d} ({pct:.2f}%) - atribuible al filtro forense F6 no replicado

## Notas sobre Bloque S (cutoff=14)

Para cutoff=14, churn_30d se calcula pero su threshold (CUTOFF+30d) cae fuera del rango observable (REFERENCE+16d). NO debe modelarse - solo churn_14d aplica.

## Outputs

- 54 archivos `data/samples/sample_user_ids_<ID>.parquet`
- `data/samples/_grid_summary.parquet`
- `informes/01_grid_samples_summary.csv`
- `informes/01_grid_samples_overview.png`

## Proximo paso

Ejecutar `02zz_rebaseline.ipynb` para reentrenar el modelo CatBoost final del TFG con cleanup dinamico.
"""

(cfg.STUDY_INFORMES / "01_grid_samples_report.md").write_text(report)
print(f'Report: {cfg.STUDY_INFORMES / "01_grid_samples_report.md"}')


Report: /Users/jezquerro/Documents/tfg/validation_study/informes/01_grid_samples_report.md
